# CSE 5243 - Introduction to Data Mining
## Homework 5: Association Analysis
- Semester: Spring 2026
- Instructor: Tom Bihari
- Section: Wed/Fri 12:45PM
- Student Name: Nicholas McCracken
- Student Email: mccracken.161@osu.edu

Template Version V2.
***

# Introduction

### Objectives

In this lab, you will use a grocery dataset provided on Carmen to find potential association rules.

The objectives of this assignment are:
- Practice the Association Analysis content we covered this semester.
- Understand “why” the particular topics, techniques, etc., are important from a practical perspective.
- Understand how to choose and use appropriate tools to solve the provided problems.

### The Dataset
- This workbook uses a market basket dataset containing transactions.  See the Excel dataset file for details.
- It is based on the French Bakery Daily Sales dataset (https://www.kaggle.com/datasets/matthieugimbert/french-bakery-daily-sales).  However, it has been reformatted and modified specifically for this asignment.  Do not publish this dataset or use it for purposes other than this assignment.
  - NOTE: Read the information about the original dataset, and review the data for issues or  considerations you might want to address.  In particular, some "Quantity" values are negative.  What does that mean?  Also, some of the Items may not be actual products (e.g., "Slice" is a separate fee for slicing bread.)  What do you choose to do with those cases?  You may choose - but you must justify your choices.
  - NOTE: For simplicity, you may set all Quantities to 1, so only one of each Item is purchased in each transaction.
- The data file captures the data in "long format". Specifically, every row corresponds to the transaction id and the item. If the specific transaction id has multiple items, there will be multiple rows in the data.
- You can process the data however you like, but it is recommended you convert into an encoded data structure suitable for use with the mlxtend package.  **This has been done for you below.**

## The Business Problem
- Assume this dataset contains all of the transactions for one month for our store.  We wish to find association rules that would improve our revenue as follows:
  - We would discount **one** of our products by **10%** each month, with the hope that this would encourage customers to visit our store to purchase that product **8%** more frequently, and also purchase other products (that are not discounted) more frequently.
- Practically speaking, we would like to come up with **two-item** rules (one antecedent and one consequent: (A -> B)) and choose the one that best adds to our revenues  (based on the rule support, confidence, etc.).
- For simplicity, don't consider complex rule interactions (e.g., A->B and B->C, A->B and B->A, etc.).  Assume each rule is completely separate.

### Proper Answers
- **IMPORTANT:** **Show your work** and **explain it**.  This will help us give partial credit in some cases.

### Collaboration
For this assignment, you should work as an individual. You may informally discuss ideas with classmates, but your work should be your own.

### What You Need to Turn In
- Submit this Jupyter Notebook in .IPYNB format.  Do not "zip" the file.

### Notes
- Feel free to use the **mlxtend** package throughout this assignment.  There are useful examples here!
  - See: **https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/#example-1-generating-association-rules-from-frequent-itemsets**
***

***
# Section: 1 - Get Ready
1A) Load the data, and get it ready for association analysis. Do this with convenient python helper methods as appropriate. Feel free to use the tools given in the example we covered. 
- Suggest: Encode the data as shown below.
***

The transaction and item data are loaded from the Excel workbook. The data is cleaned and converted into a one-hot encoded format suitable for analysis of the associations between items.

Preprocessing steps:
- Remove rows with missing transaction, item, or quantity values.
- Remove non-positive quantities, since negative values are likely to represent returns or corrected mistakes (false purchases) and zero does not represent a purchase.
- Set all remaining item quantities to 1 (binary), since this analysis only cares whether an item was purchased, not how many times it was purchased.
- Remove duplicate item pairs so that each item is only appearing at most once for per transaction.

In [83]:
import numpy as np
import pandas as pd
import mlxtend as mlx
from mlxtend.frequent_patterns import association_rules, apriori, fpgrowth
from mlxtend.preprocessing import TransactionEncoder
from collections import defaultdict

pd.set_option('display.max_columns', 1000)

# Business problem parameters
price_discount  = 0.10
purchase_uplift = 0.08
data_file_name = 'Sp26_TEB_French_Bakery_V2.xlsx'

# Load transaction and item data
transaction_df = pd.read_excel(data_file_name, sheet_name = 'TxToItem')
item_df = pd.read_excel(data_file_name, sheet_name = 'Items')

# Remove rows with missing key values
transaction_df = transaction_df.dropna(subset=['TxID', 'ItemID', 'Quantity'])
item_df = item_df.dropna(subset=['ItemID'])

# Keep only actual purchases, then convert quantity to binary (1 for purchase, 0 for no purchase)
# Only relevant whether an item was purchased or not, not how many were purchased
transaction_df = transaction_df[transaction_df['Quantity'] > 0].copy()
transaction_df['Quantity'] = 1

# Remove duplicate item occurrences within the same transaction
transaction_df = transaction_df.drop_duplicates(subset=['TxID', 'ItemID']).copy()

# Merge item names into the transaction data for human-readable analysis of rules
item_name_col = item_df.columns[1]
item_lookup_df = item_df[['ItemID', item_name_col]].drop_duplicates().copy()
item_lookup_df.columns = ['ItemID', 'ItemName']

transaction_item_df = transaction_df.merge(item_lookup_df, on='ItemID', how='left')

# Remove 'Slice' items; this is only a fee for slicing bread and not an actual item purchase
transaction_item_df = transaction_item_df[transaction_item_df['ItemName'] != 'Slice'].copy()

# Encode the transaction data using item names
transaction_items = defaultdict(list)
for transaction in transaction_item_df[['TxID', 'ItemName']].values.tolist():
    transaction_items[transaction[0]].append(transaction[1])

dataset_modified = list(transaction_items.values())

te = TransactionEncoder()
te_ary = te.fit(dataset_modified).transform(dataset_modified)
encoded_transaction_df = pd.DataFrame(te_ary, columns=te.columns_)

In [84]:
# Dataset summary
display(transaction_df.head())
display(item_df.head())
display(transaction_item_df.head())
display(encoded_transaction_df.head(5))

,TxID,ItemID,Quantity
0,150040,5,1
1,150040,86,1
2,150041,86,1
3,150041,85,1
4,150042,141,1


,ItemID,Name,UnitPrice,Category,Subcategory
0,1,Unknown,0.00,Other,Unclassified
1,2,12 Macarons,11.70,Confectionery & Snacks,Small Sweets
2,3,Armorican Cake,2.92,Pastry & Desserts,Cakes (Whole)
3,4,Article 295,0.00,Other,Unclassified
4,5,Baguette,1.05,Bread,Artisan & Daily Bread


,TxID,ItemID,Quantity,ItemName
0,150040,5,1,Baguette
1,150040,86,1,Chocolate Croissant
2,150041,86,1,Chocolate Croissant
3,150041,85,1,Bread
4,150042,141,1,Traditional Baguette


,12 Macarons,Almond Croissant,Almond Financier,Aperitif Baguette,Apple Galette 4 persons,Apple Galette 6 persons,Apple Turnover,Apricot Flan,Armorican Cake,Article 295,Assorted Bakery Items,Assorted Confectionery,Assorted Drinks,Assorted Pastries,Assorted Sandwiches,Assorted Tartlets,Assorted Viennoiserie,Bag of Croutons,Bag of Viennoiserie,Baguette,Banette Bread,Bottereau Fritter,Bread,Breton Shortbread,Brioche,Brioche Loaf,Brownies,Cake,Caramel Walnut Cake,Caramel or Pistachio Crumble,Catering,Cereal Baguette,Chocolate,Chocolate Almond Bread,Chocolate Croissant,Chocolate Fondant,Chocolate Tartlet,Christmas Brioche,Coffee or Water,Cookie,Country Bread,Cream Puff,Croissant,Crumble,Custard Flan,Drink 33cl,Eclair,Financier Pack of 5,Frangipane Galette 4 persons,Frangipane Galette 6 persons,Fruit Tart 4 persons,Fruit Tart 6 persons,Galette 8 persons,Guerande Cake,Half Baguette,Half Loaf,Harvest Bread,JB Sandwich,JB Sandwich with Emmental,Japanese Walnut Cake,Kouign-Amann,Large Breton Custard,Large Kouign-Amann,Large Lollipop,Large Nantais Cake,Large Savory Platter,Layered Dessert,Lollipop,Macaron,Meal,Meal 6.50,Meal 7.00,Meal 7.60,Meal 8.30,Meringue,Mille-feuille,Mini Brioche,Nantais Cake,Nest Pastry,Palmier,Paris-Brest,Pasta,Pasta Meal Deal,Pear Chocolate Galette 4 persons,Pear Chocolate Galette 6 persons,Polka Bread,Prepared Meal 5.50,Prepared Meal 6.00,Prepared Meal 6.50,Prepared Meal 7.00,Prepared Meal Deal,Quim Bread,Raisin Swirl,Raspberry Cake,Raspberry Tropézienne,Religious Pastry,Round Bread 200g,Round Bread 400g,Royal Cake,Royal Cake 4 persons,Royal Cake 6 persons,Rye Bread,SABLE F P,Saint Honore Cake,Salt-Free Bread,Sandwich Bread,Sandwich Meal Deal,Savarin,Seed Bread,Seeded Baguette,Small Banette,Small Nantais Cake,Small Savory Platter,Special Bread,Special Bread per kg,Straw,Strawberry Cake,Strawberry Pistachio Eclair,Strawberry Tart 4 persons,Strawberry Tart 6 persons,Strawberry Tartlet,Sweet Mini Pastries 12,Sweet Mini Pastries 24,Swiss Chocolate Bread,Tart Tray 25 persons,Tartlet,Tea,Thin Baguette,Thin Tart,Three Chocolate Cake,Traditional Baguette,Triangle Pastries,Tropical Delight,Tropézienne,Tulip Pastry,Unknown,Vienna Bread,Viking Bread,Whole Sandwich,Whole Wheat Bread,Winter Sweet,Yule Log Cake 4 persons,Yule Log Cake 6 persons,Yule Log Cake 8 persons
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False

***
# Section: 2 - Explore the Data
***

***
## Section: 2.1 - Get the Transaction and Item Sizes
- Calculate the **number_of_transactions** and **number_of_items**.
***

In [85]:
number_of_items = encoded_transaction_df.shape[1]
number_of_transactions = len(dataset_modified)

print("Number of items:", number_of_items)
print("Number of transactions:", number_of_transactions)

Number of items: 144
Number of transactions: 135677


***
## Section: 2.2 - Evaluate the Itemset and Rule Size & Complexity
- Calculate the **maximum number of Itemsets** that could be created from the items (without considering the actual transaction data). Show your work.
- Calculate the **maximum number of Rules** that can be created from the items (without considering the actual transaction data). Show your work.
- What do the calculations suggest as a **potential cause of concern**? Hint: Complexity.
- What might you do to manage these concerns?
***

The processed dataset contains 135,686 transactions and 145 items. 

Since this dataset has 145 items, the maximum number of possible **Itemsets** is:

$$
2^{145} - 1 \approx 4.46 \times 10^{43}
$$

The maximum number of possible **Rules** is:

$$
3^{145} - 2^{146} + 1 \approx 1.52 \times 10^{87}
$$

Both values are extremely large, which shows that this analysis can become very expensive to compute very quickly. The exponential growth of the search space as the number of items in the dataset increases would be the main concern.

For managing this complexity, we could reduce said search space. This can be done by limiting the size of Itemsets, shifting focus to only two item rules, improving the efficiency of the given enumeration algorithm, or adding a threshold for the minimum support.

In [86]:
max_itemsets = (2 ** number_of_items) - 1
max_rules = (3 ** number_of_items) - (2 ** (number_of_items + 1)) + 1

print("Maximum possible number of itemsets:", max_itemsets)
print("Maximum possible number of rules:", max_rules)

Maximum possible number of itemsets: 22300745198530623141535718272648361505980415
Maximum possible number of rules: 507528786056415600719754115140205959847495967120227341966904430154050


**Discussion:**

***
# Section: 3 - Itemset Generation
- Create a set of 20 two-item sets with highest support. Sort them in decreasing order of support.
- NOTE: You can set the **apriori** function to create itemsets of **max_len=2** and try various values for **min_support** to get the right number of 2-itemsets.  Then you can filter them for only the 2-itemsets.  But keep the 1and2-itemsets and the 1-itemsets around - they might be useful later. 
- Show the results, briefly.
- Explain what you did and why you did it.
***

In [87]:
# Generate frequent itemsets (1-item and 2-item)
frequent_itemsets = apriori(
    encoded_transaction_df,
    min_support=0.01,
    use_colnames=True,
    max_len=2
)

# Filter only 2-item itemsets
two_itemsets = frequent_itemsets[frequent_itemsets['itemsets'].apply(lambda x: len(x) == 2)]

# Sort by support and take top 20
top_20_itemsets = two_itemsets.sort_values(by='support', ascending=False).head(20)

display(top_20_itemsets)

,support,itemsets
34,0.039565,"frozenset({Chocolate Croissant, Croissant})"
36,0.036115,"frozenset({Traditional Baguette, Croissant})"
35,0.030867,"frozenset({Chocolate Croissant, Traditional Ba..."
32,0.014955,"frozenset({Baguette, Traditional Baguette})"
37,0.011763,"frozenset({Viking Bread, Traditional Baguette})"
33,0.010857,"frozenset({Traditional Baguette, Banette Bread})"
31,0.010835,"frozenset({Baguette, Croissant})"


| min_support | # 2-itemsets  | Notes |
|-------------|----------------------|------|
| 0.05        | 0               | Too restrictive to find any frequent 2-itemsets |
| 0.01        | 7   | Captures only the most frequent item combinations |
| 0.005       | 20+      | Adds a few additional lower-support but still meaningful pairs |
| 0.001       | 20+     | Adds a few additional lower-support but still meaningful pairs |

Multiple minimum support thresholds were tested (0.05, 0.01, 0.005, and 0.001). The results above show that the top frequent itemsets remain largely unchanged across these values. Lowering the minimum support threshold introduces some additional lower-frequency itemsets, but overall does not improve the quality of the most important item pairing patterns.

Using a minimum support of 0.01 is best, since it captures the strongest and most meaningful associations while avoiding unnecessary computational complexity from weaker itemsets created by lower minimum support thresholds. Any higher threshold will result in no 2-item itemsets being found.

***
# Section: 4 - Generate Rules
- For the two-itemsets created above, create the related rules.
- Use the **association_rules** function.  You need to pass in the 1and2_itemsets to the function.  If you set **min_threshold=0.0**, you will get all of the rules.
***

Rules were generated from the frequent itemsets using confidence, and only two-item rules  were kept. The rules were then sorted by confidence to identify the strongest relationships.

In [88]:
# Generate rules from frequent itemsets
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.0   # get ALL rules
)

# Keep only 2-item rules
rules = rules[
    (rules['antecedents'].apply(len) == 1) & 
    (rules['consequents'].apply(len) == 1)
]

# Sort by confidence for display
rules = rules.sort_values(by='confidence', ascending=False)
display(rules.head(10))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
12,frozenset({Viking Bread}),frozenset({Traditional Baguette}),0.022915,0.495264,0.011763,0.513348,1.036514,1.0,0.000414,1.037160,0.036053,0.023228,0.035828,0.268550
6,frozenset({Chocolate Croissant}),frozenset({Croissant}),0.077235,0.083927,0.039565,0.512263,6.103649,1.0,0.033082,1.878209,0.906150,0.325373,0.467578,0.491839
7,frozenset({Croissant}),frozenset({Chocolate Croissant}),0.083927,0.077235,0.039565,0.471415,6.103649,1.0,0.033082,1.745726,0.912770,0.325373,0.427172,0.491839
11,frozenset({Croissant}),frozenset({Traditional Baguette}),0.083927,0.495264,0.036115,0.430315,0.868860,1.0,-0.005451,0.885991,-0.141456,0.066501,-0.128680,0.251618
8,frozenset({Chocolate Croissant}),frozenset({Traditional Baguette}),0.077235,0.495264,0.030867,0.399656,0.806956,1.0,-0.007384,0.840744,-0.205876,0.056990,-0.189422,0.230991
2,frozenset({Baguette}),frozenset({Traditional Baguette}),0.112082,0.495264,0.014955,0.133425,0.269402,1.0,-0.040556,0.582449,-0.753345,0.025244,-0.716889,0.081810
1,frozenset({Croissant}),frozenset({Baguette}),0.083927,0.112082,0.010835,0.129095,1.151783,1.0,0.001428,1.019534,0.143854,0.058510,0.019160,0.112880
0,frozenset({Baguette}),frozenset({Croissant}),0.112082,0.083927,0.010835,0.096666,1.151783,1.0,0.001428,1.014102,0.148416,0.058510,0.013906,0.112880
5,frozenset({Banette Bread}),frozenset({Traditional Baguette}),0.129211,0.495264,0.010857,0.084023,0.169652,1.0,-0.053137,0.551035,-0.848958,0.017693,-0.814767,0.052972
10,frozenset({Traditional Baguette}),frozenset({Croissant}),0.495264,0.083927,0.036115,0.072921,0.868860,1.0,-0.005451,0.988128,-0.230198,0.066501,-0.012015,0.251618


The strongest rules involve relationships between croissant products:

- Chocolate Croissant → Croissant (confidence ~0.51, lift ~6.10)  
- Croissant → Chocolate Croissant (confidence ~0.47, lift ~6.10)  

These rules have high lift values, indicating that purchasing one type of croissant significantly increases the likelihood of purchasing the other, suggesting a complementary relationship between the two products. This goes along with our obvious human intuition. 

Some other notable relationship include:

- Viking Bread → Traditional Baguette (confidence ~0.51, lift ~1.04)  
- Croissant → Traditional Baguette (confidence ~0.43, lift < 1)  

Many of these rules have lift values close to or below 1, which actually indicates weak or even negative associations. This suggests that Traditional Baguette is a very commonly purchased item, and its presence in the rules does not necessarily indicate the customer is likely to purchase any other item.

Overall, the most meaningful rules are those with both high confidence and high lift, particularly the relationships between croissant products. These rules can be used to help make decisions in the proceeding section.

***
# Section: 5 - Rule Evaluation
- For the rules created above, find the single Item (that would be given the discount) that would cause the greatest increase in monthly store revenue.
  - This is based on the Business Problem stated at the top of this notebook.
  - Consider:
    - How much will the store's monthly revenue decrease (or increase) due to the change in price for the chosen Item (and its increased sales)?
    - How much will the store's monthly revenue increase (or decrease) due to the increased sales of the associated Items?
***

The rules were evaluated based on their estimated impact on the store'smonthly revenue, given the assumption in the business problem that discounting a product by 10% increases its purchases by 8%.

For each 2-item rule of the form A -> B, the following were considered:
- revenue loss from discounting item A,
- revenue gain from increased purchases of A,
- revenue from increased purchases of item B, estimated using the rule’s confidence from section 4

In [90]:
# Merge in item prices
item_price_col = item_df.columns[2]  # price is third column
item_prices = item_df[['ItemID', item_price_col]].copy()
item_prices.columns = ['ItemID', 'Price']

# Map item names to prices
item_name_col = item_df.columns[1]
item_lookup = item_df[['ItemID', item_name_col]].copy()
item_lookup.columns = ['ItemID', 'ItemName']

item_price_lookup = item_lookup.merge(item_prices, on='ItemID')
prices = dict(zip(item_price_lookup['ItemName'], item_price_lookup['Price']))

# Evaluate each rule by computing it's impact on the store's monthly revenue
evaluations = []
for _, row in rules.iterrows():  # use iterrows to access row values (to work with dfs)
    A = list(row['antecedents'])[0]
    B = list(row['consequents'])[0]
    antecedent_support = row['antecedent support']
    confidence = row['confidence']

    if A not in prices or B not in prices: continue

    price_A = prices[A]
    price_B = prices[B]

    # Baseline number of purchases of A
    count_A = antecedent_support * number_of_transactions

    # Extra purchases of A caused by discount
    additional_A = count_A * purchase_uplift

    # Revenue loss from discounting all A purchases
    loss_from_discount = count_A * price_A * price_discount

    # Revenue from extra purchases of A at the discounted price
    revenue_from_additional_A = additional_A * price_A * (1 - price_discount)

    # Extra purchases of B that are induced by the extra A purchases
    additional_B = additional_A * confidence
    revenue_from_additional_B = additional_B * price_B

    net_revenue = revenue_from_additional_A + revenue_from_additional_B - loss_from_discount

    evaluations.append({
        'Rule': f"{A} -> {B}",
        'Revenue Impact': net_revenue
    })

# Sort by net revenue impact for display
evaluations_df = pd.DataFrame(evaluations)
evaluations_df = evaluations_df.sort_values(by='Revenue Impact', ascending=False)
display(evaluations_df.head(10))

,Rule,Revenue Impact
2,Croissant -> Chocolate Croissant,189.91756
1,Chocolate Croissant -> Croissant,143.20080
3,Croissant -> Traditional Baguette,137.50156
4,Chocolate Croissant -> Traditional Baguette,58.27920
0,Viking Bread -> Traditional Baguette,-75.43984
5,Baguette -> Traditional Baguette,-219.83780
6,Croissant -> Baguette,-287.81844
7,Baguette -> Croissant,-295.38180
8,Banette Bread -> Traditional Baguette,-640.04752
9,Traditional Baguette -> Croissant,-2128.40320




The results show that:

- The rule Croissant -> Chocolate Croissant produces the highest revenue gain (~189.92), making it the best candidate for discounting.
- The inverse rule Chocolate Croissant -> Croissant also performs strongly (~143.20), which further speaks to the strong relationship between these two items.
- Rules involving Traditional Baguette as the B often show positive revenue impact, but less than the croissant-based rules.
- Many rules produce negative revenue impact, especially when discounting lower-margin items like the Traditional Baguette. The revenue lost from discounting outweighs any gains from increased purchases.

The store should discount the Croissant, as this is expected to produce the largest increase in overall revenue due to both increased Croissant purchases and additional purchases of related items like the Chocolate Croissant.

***
# Section: 7 - Conclusions
- Write a paragraph on what you discovered or learned from this homework.
***

This homework highlighted the importance of how analysis between the relationship rules of items can uncover meaningful patterns in transaction data, which can then be applied to improve the revenue outcomes of a business. 

Using the Apriori algorithm, frequent itemsets were identified, and rules were generated and evaluated using metrics such as support, confidence, and lift. The results showed that not all rules with high support are guarenteed to be useful. Lift is an even more important measure for identifying actual meaningful relationships between store items.

One key takeaway I had is that the number of possible itemsets and rules grows exponentially as the number of items increases. This makes it important to limit the search spaces, and is why I chose a larger  minimum support threshold to filter out unimportant rules.

Overall, this homework further deepened my understanding in how to combine data mining techniques with business context to make effective and actually impactful real-world decisions..

***
### END-OF-SUBMISSION
***